In [1]:
import pandas as pd
import numpy as np
#from torch.utils.data import Dataset, DataLoader
#from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv("dataset.csv")
data.head()

,target_x,target_y,target_z,qx,qy,qz,qw,joint_0,joint_1,joint_2,joint_3,joint_4
0,-0.2,-0.19397,-0.05,0.769304,0.565324,-0.294709,0.041548,-0.795495,1.428087,0.178582,0.931122,-2.062491
1,-0.2,-0.19397,-0.05,0.816882,0.494089,-0.289798,0.067798,-0.795627,1.428643,0.177629,0.931517,-1.883144
2,-0.2,-0.19397,-0.05,0.857884,0.418875,-0.282554,0.093503,-0.795760,1.429339,0.176359,0.932091,-1.703798
3,-0.2,-0.19397,-0.05,0.891978,0.340288,-0.273035,0.118454,-0.795892,1.429983,0.175137,0.932669,-1.524452
4,-0.2,-0.19397,-0.05,0.918890,0.258962,-0.261318,0.142452,-0.796024,1.430169,0.174839,0.932780,-1.345106


In [3]:
pd.options.display.float_format = lambda x: f"{x:.3f}".rstrip('0').rstrip('.')
data.describe()

,target_x,target_y,target_z,qx,qy,qz,qw,joint_0,joint_1,joint_2,joint_3,joint_4
count,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411
mean,-0.143,-0.03,0.034,0.583,-0.059,-0.035,0.052,-0.233,0.465,-0.367,0.863,-0.188
std,0.048,0.152,0.056,0.53,0.532,0.16,0.254,0.798,0.602,0.848,1.07,0.974
min,-0.2,-0.2,-0.05,-0.997,-1,-0.569,-0.542,-2.32,-0.738,-1.911,-0.931,-2.208
25%,-0.182,-0.166,-0.014,0.544,-0.543,-0.129,-0.151,-0.847,-0.179,-1.433,-0.459,-0.986
50%,-0.156,-0.102,0.025,0.79,-0.051,-0.028,0.069,-0.514,0.685,0.18,1.526,-0.265
75%,-0.114,0.154,0.075,0.911,0.398,0.034,0.219,0.648,0.987,0.297,1.74,0.627
max,-0.011,0.2,0.15,1,1,0.687,0.707,1.812,1.53,1.075,1.873,2.311


In [4]:
data = data.rename(columns={"joint_0": "Root.001_ry", 
                     "joint_1": "Root.002_rx", 
                     "joint_2": "Root.003_rx",
                     "joint_3": "Root.004_rx",
                     "joint_4": "Head_ry",
                    })

In [5]:
# Normalizzazione quaternioni
q = data[["qx", "qy", "qz", "qw"]].values
norm = np.linalg.norm(q, axis=1, keepdims=True)

data[["qx", "qy", "qz", "qw"]] = q / norm

In [6]:
# Inversione quaternioni uguali per non fare la media tra i due
#mask = data["qw"] < 0

#cols = ["qx", "qy", "qz", "qw"]
#data.loc[mask, cols] = -data.loc[mask, cols]

In [9]:
print((data["qw"] < 0).sum()) #dev'essere 0 se ha funzionato l'inversione

0


In [10]:
cols = ["Root.001_ry", "Root.002_rx", "Root.003_rx", "Root.004_rx", "Head_ry"]

for col in cols:
    data[f"{col}_sin"] = np.sin(data[col])
    data[f"{col}_cos"] = np.cos(data[col])
data.head()

,target_x,target_y,target_z,qx,qy,qz,qw,Root.001_ry,Root.002_rx,Root.003_rx,...,Root.001_ry_sin,Root.001_ry_cos,Root.002_rx_sin,Root.002_rx_cos,Root.003_rx_sin,Root.003_rx_cos,Root.004_rx_sin,Root.004_rx_cos,Head_ry_sin,Head_ry_cos
0,-0.2,-0.194,-0.05,0.769,0.565,-0.295,0.042,-0.795,1.428,0.179,...,-0.714,0.7,0.99,0.142,0.178,0.984,0.802,0.597,-0.882,-0.472
1,-0.2,-0.194,-0.05,0.817,0.494,-0.29,0.068,-0.796,1.429,0.178,...,-0.714,0.7,0.99,0.142,0.177,0.984,0.803,0.597,-0.952,-0.307
2,-0.2,-0.194,-0.05,0.858,0.419,-0.283,0.094,-0.796,1.429,0.176,...,-0.714,0.7,0.99,0.141,0.175,0.984,0.803,0.596,-0.991,-0.133
3,-0.2,-0.194,-0.05,0.892,0.34,-0.273,0.118,-0.796,1.43,0.175,...,-0.714,0.7,0.99,0.14,0.174,0.985,0.803,0.596,-0.999,0.046
4,-0.2,-0.194,-0.05,0.919,0.259,-0.261,0.142,-0.796,1.43,0.175,...,-0.715,0.7,0.99,0.14,0.174,0.985,0.803,0.596,-0.975,0.224


In [12]:
data = data.rename(columns={"qx": "hand_quat_qx",
                     "qy": "hand_quat_qy",
                     "qz": "hand_quat_qz",
                     "qw": "hand_quat_qw"
                    })

In [13]:
data.describe()

,target_x,target_y,target_z,hand_quat_qx,hand_quat_qy,hand_quat_qz,hand_quat_qw,Root.001_ry,Root.002_rx,Root.003_rx,...,Root.001_ry_sin,Root.001_ry_cos,Root.002_rx_sin,Root.002_rx_cos,Root.003_rx_sin,Root.003_rx_cos,Root.004_rx_sin,Root.004_rx_cos,Head_ry_sin,Head_ry_cos
count,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,...,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411,21741411
mean,-0.143,-0.03,0.034,0.083,-0.254,-0.011,0.214,-0.233,0.465,-0.367,...,-0.187,0.686,0.384,0.733,-0.191,0.643,0.445,0.249,-0.109,0.585
std,0.048,0.152,0.056,0.783,0.471,0.163,0.147,0.798,0.602,0.848,...,0.666,0.227,0.52,0.21,0.607,0.426,0.722,0.467,0.702,0.391
min,-0.2,-0.2,-0.05,-1,-1,-0.569,0,-2.32,-0.738,-1.911,...,-1,-0.681,-0.673,0.041,-1,-0.333,-0.802,-0.298,-0.999,-0.674
25%,-0.182,-0.166,-0.014,-0.762,-0.659,-0.102,0.081,-0.847,-0.179,-1.433,...,-0.749,0.599,-0.178,0.551,-0.99,0.138,-0.443,-0.168,-0.834,0.394
50%,-0.156,-0.102,0.025,0.435,-0.338,-0.018,0.203,-0.514,0.685,0.18,...,-0.491,0.723,0.633,0.773,0.18,0.941,0.969,0.044,-0.262,0.692
75%,-0.114,0.154,0.075,0.864,0.094,0.055,0.315,0.648,0.987,0.297,...,0.604,0.837,0.834,0.928,0.293,0.972,0.992,0.793,0.587,0.902
max,-0.011,0.2,0.15,1,1,0.687,0.707,1.812,1.53,1.075,...,1,0.999,0.999,1,0.879,1,1,1,1,1


In [14]:
data.to_csv("dataset_fixed.csv", index=False)

In [16]:
data.columns

Index(['target_x', 'target_y', 'target_z', 'hand_quat_qx', 'hand_quat_qy',
       'hand_quat_qz', 'hand_quat_qw', 'Root.001_ry', 'Root.002_rx',
       'Root.003_rx', 'Root.004_rx', 'Head_ry', 'Root.001_ry_sin',
       'Root.001_ry_cos', 'Root.002_rx_sin', 'Root.002_rx_cos',
       'Root.003_rx_sin', 'Root.003_rx_cos', 'Root.004_rx_sin',
       'Root.004_rx_cos', 'Head_ry_sin', 'Head_ry_cos'],
      dtype='str')

In [4]:
cols = ['Root.001_ry', 'Root.002_rx',
        'Root.003_rx', 'Root.004_rx', 'Head_ry']

data[cols] = np.degrees(data[cols])
data.head()

,target_x,target_y,target_z,hand_quat_qx,hand_quat_qy,hand_quat_qz,hand_quat_qw,Root.001_ry,Root.002_rx,Root.003_rx,...,Root.001_ry_sin,Root.001_ry_cos,Root.002_rx_sin,Root.002_rx_cos,Root.003_rx_sin,Root.003_rx_cos,Root.004_rx_sin,Root.004_rx_cos,Head_ry_sin,Head_ry_cos
0,-0.2,-0.19397,-0.05,0.769304,0.565324,-0.294709,0.041548,-45.578517,81.823357,10.232001,...,-0.714210,0.699931,0.989834,0.142225,0.177634,0.984097,0.802290,0.596934,-0.881534,-0.472120
1,-0.2,-0.19397,-0.05,0.816882,0.494089,-0.289798,0.067798,-45.586095,81.855225,10.177401,...,-0.714303,0.699837,0.989913,0.141675,0.176697,0.984265,0.802526,0.596617,-0.951615,-0.307294
2,-0.2,-0.19397,-0.05,0.857884,0.418875,-0.282554,0.093503,-45.593674,81.895097,10.104619,...,-0.714395,0.699742,0.990012,0.140986,0.175446,0.984489,0.802868,0.596156,-0.991168,-0.132610
3,-0.2,-0.19397,-0.05,0.891978,0.340288,-0.273035,0.118454,-45.601253,81.932002,10.034622,...,-0.714488,0.699648,0.990102,0.140348,0.174243,0.984703,0.803213,0.595693,-0.998926,0.046328
4,-0.2,-0.19397,-0.05,0.918890,0.258962,-0.261318,0.142452,-45.608830,81.942642,10.017540,...,-0.714580,0.699553,0.990128,0.140164,0.173950,0.984755,0.803279,0.595603,-0.974640,0.223779


In [5]:
data.to_csv("dataset_fixed_deg.csv", index=False)